### Fact table creation fact orders

In [0]:
#data reading 
df_orders = spark.sql("select * from databricks_cata.silver.orders_silver")
df_orders.display()               

In [0]:
df_product = spark.sql("select * from databricks_cata.gold.dimproducts")
df_product.display()


In [0]:
#dimension  dataframe
df_dimcustomers = spark.sql("select DimCustomerKey, customer_id as dim_customer_key from databricks_cata.gold.dimcustomers")

df_dimproducts = spark.sql("select product_id as DimProductKey, product_id as dim_product_id from databricks_cata.gold.dimproducts")

In [0]:
# fact dataframe
df_fact = df_orders.join(df_dimcustomers, df_orders['customer_id'] == df_dimcustomers['dim_customer_key'], how='left').join(df_dimproducts, df_orders['product_id'] == df_dimproducts['dim_product_id'], how='left')

df_fact = df_fact.drop('customer_id', 'product_id', 'dim_customer_key', 'dim_product_id')
df_fact.display()



In [0]:
#Upsert on fact table 

from delta.tables import DeltaTable

if spark.catalog.tableExists("databricks_cata.gold.FactOrders"):

    delta_table_object = DeltaTable.forName(spark, "databricks_cata.gold.FactOrders")

    delta_table_object.alias("trg").merge(df_fact.alias("src"), 
                                          "trg.order_id = src.order_id AND trg.DimCustomerKey = src.DimCustomerKey AND trg.DimProductKey = src.DimProductKey")\
                                              .whenMatchedUpdateAll()\
                                                  .whenNotMatchedInsertAll()\
                                                      .execute()
                                                    
else:

    df_fact.write.format("delta")\
        .option("path", "abfss://gold@monarchazuredatalake.dfs.core.windows.net/FactOrders")\
            .saveAsTable("databricks_cata.gold.FactOrders")




######checking the final fact table data 


In [0]:
%sql
select  * from databricks_cata.gold.Factorders